# L6a: Introduction to Single Index Models (SIMs)
In this lecture, we replace the full covariance matrix of L5a and L5b, with its one entry per pair of assets, by a model with one common factor. A single index model (SIM) says that every asset's growth rate is a linear function of the growth rate of a market index plus a residual that is specific to the asset. That one line gives each asset a __beta__ (its exposure to the market), splits its risk into a systematic part and an idiosyncratic part, and, under an assumption we will state and check, gives the whole covariance matrix from a handful of numbers per asset. We estimate the model by least squares, quantify how uncertain the estimates are, and say what the estimates do and do not prove; L6b builds portfolios from them.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Decompose an asset's growth rate with a single index model:__ Write the SIM and its assumptions, interpret the intercept, the beta, and the residual, and split an asset's growth-rate variance into a market part and an asset-specific part.
> * __Build the covariance a SIM implies and read portfolio risk from it:__ Derive the rank-one-plus-diagonal covariance matrix, count what it saves, write a portfolio's beta and variance under the model, and state the residual assumption that must be checked before the covariance is used.
> * __Estimate and diagnose a SIM:__ Set up the least-squares regression, compute the estimates and their classical uncertainty in growth-rate units, and use the coefficient of determination and residual diagnostics without overstating what they establish.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Estimate single index models from historical data](CHEME-5660-L6a-Example-SVD-SIM-Estimation-Fall-2026.ipynb). Build the growth-rate matrix from the 2014 to 2024 data, fit the SIM for one firm by least squares three equivalent ways, compute its classical uncertainty and fit statistics next to those of an index fund, then fit every security in the dataset, check the residual-correlation assumption the SIM covariance relies on, and save the parameter archive that L6b builds portfolios from.

The second example asks how much the estimates move:

> [▶ Bootstrap uncertainty in a single index model](CHEME-5660-L6a-Example-SIM-Parameter-Uncertainty-Fall-2026.ipynb). Fit the SIM with the course package, resample it two ways (empirical residuals and Gaussian innovations), compare the bootstrap intervals with the classical standard errors, audit the residuals for the heavy tails and dependence that every independent-draws method ignores, and compute the autocorrelation-consistent standard errors that dependence calls for.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: Data Driven Minimum Variance Portfolio Allocation
Last time we introduced Modern Portfolio Theory (MPT) and minimum-variance portfolio allocation. Let's quickly review the key concepts.

> __Key Idea:__ The key idea of minimum-variance portfolio allocation is to balance risk and reward by diversifying across assets whose growth rates are less than perfectly correlated. The minimum-variance portfolio for a target growth rate is the portfolio that minimizes the variance of the portfolio growth rate while meeting the target.

Consider a portfolio $\mathcal{P}$ of risky assets (assets with uncertain growth rates, such as equities and ETFs; the horizon-matched risk-free asset of L5b is not among them). The linearized one-period portfolio growth rate is $g_{p}=\mathbf{w}^{\top}\mathbf{g}$ (exact to first order in the time step; L5b), with reward $\mathbb{E}[g_{p}]=\mathbf{w}^{\top}\bar{\mathbf{g}}$ and risk $\text{Var}(g_{p})=\mathbf{w}^{\top}\mathbf{\Sigma}_{g}\mathbf{w}$, where $\bar{\mathbf{g}}$ is the mean growth-rate vector and $\mathbf{\Sigma}_{g}$ the growth-rate covariance matrix (L5a, L5b). The long-only optimal weights $\mathbf{w}$ for a target growth rate $g_{\star}$ solve:
$$
\boxed{
\begin{align*}
\text{minimize}~\text{Var}(g_{p}) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}\underbrace{\text{Cov}\left(g_{i},g_{j}\right)}_{=\;\rho_{ij}\sqrt{\Sigma_{g,ii}\Sigma_{g,jj}}}\quad{\Longleftrightarrow\mathbf{w}^\top \mathbf{\Sigma}_{g} \mathbf{w}} \\
\text{subject to}~\mathbb{E}(g_{p})& =  \sum_{i\in\mathcal{P}}w_{i}\;\mathbb{E}(g_{i})\geq g_{\star}\quad\Longleftrightarrow\mathbf{w}^\top \bar{\mathbf{g}} \geq g_{\star} \\
\sum_{i\in\mathcal{P}}w_{i} & =  1 \\
w_{i} & \geq  0\quad\forall{i}\in\mathcal{P}
\end{align*}}
$$
The term $g_{\star}$ is the target growth rate for portfolio $\mathcal{P}$ specified by the investor, and $\rho_{ij}$ is the correlation of the growth rates of assets $i$ and $j$. Because the target enters as a floor, sweeping $g_{\star}$ returns the global minimum-variance portfolio and then the efficient branch of the frontier, sketched in the figure: the curve of minimum-variance portfolios, the global minimum-variance portfolio at its vertex, the efficient upper branch, and the dominated lower branch.

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-Schematic.png" width="620" alt="Schematic of the minimum-variance frontier in the risk-growth plane: a curve opening to the right with the global minimum-variance portfolio at its vertex, the efficient upper branch drawn solid through a portfolio labeled 1, the dominated lower branch dashed through a portfolio labeled 3 with the same risk as portfolio 1 and lower expected growth, and the GMV portfolio labeled 2"/>
    </center>
</div>

Every entry of this problem comes from data: the mean growth rates and the covariance matrix. L5a estimated them directly, one covariance per pair of assets, and L5b's advanced material showed how noisy those pairwise estimates are and how much the optimizer amplifies the noise. Today's question is whether a model with far fewer parameters can supply the same inputs. If we did not finish the L5b example, we pick it up now.

> __Example__
>
> [▶ Compute minimum-variance portfolios, the efficient frontier, and the capital allocation line from data](../../week-5/L5b/CHEME-5660-L5b-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). Estimate the inputs for a chosen set of firms from the 2014 to 2024 data, compute the global minimum-variance portfolio in closed form and with a long-only solver, sweep the target growth rate to trace the efficient frontier, find the tangent portfolio and the capital allocation line, and compare the optimized portfolios with equal weights and an index fund on 2025 data the optimizer never saw.

___

## Factor Models
A portfolio $\mathcal{P}$ of $|\mathcal{P}|$ assets needs $|\mathcal{P}|$ mean growth rates and $|\mathcal{P}|(|\mathcal{P}|+1)/2$ distinct covariance entries; the second number grows with the square of the first, and with a few years of daily data the pairwise estimates are noisy. Yet most securities visibly move together when market-wide news arrives. A __factor model__ makes that shared source of movement explicit: it writes each asset's growth rate as a linear function of a few common factors plus an asset-specific residual, so that the co-movement between assets is carried by the factors, and only the loadings on the factors and the residual variances have to be estimated.

Let $g_{i,t}$ be the growth rate of asset $i$ on trading day $t$ (units: inverse years; the time index is a subscript, as in the L5a data matrix), let $g_{M,t}$ be the growth rate of a __market index__ $M$ on the same day (in this course the SPDR S&P 500 ETF, ticker SPY, which tracks a market-capitalization-weighted index of large U.S. firms; the letter $M$ names the index, and we write the number of assets as $|\mathcal{P}|$ in this lecture to keep the two apart), and let $f_{1,t},\dots,f_{K,t}$ be $K$ further factors such as interest-rate, inflation, or size and value spreads. A linear factor model gives each asset an intercept $\alpha_{i}$ (units: inverse years), a sensitivity $\beta_{i}$ (dimensionless) to the market index, a sensitivity $\gamma_{ik}$ to each further factor $k$, and a residual $\varepsilon_{i,t}$ (units: inverse years), the part of the growth rate the factors do not explain, and reads:
$$
\begin{align*}
g_{i,t} &= \underbrace{\alpha_{i}}_{\text{intercept}} + \underbrace{\beta_{i}\,g_{M,t}}_{\text{market}} + \overbrace{\sum_{k=1}^{K}\gamma_{ik}\,f_{k,t}}^{\text{other factors}} + \varepsilon_{i,t}
\end{align*}
$$
The co-movement between assets is carried by the factors; the residuals are what is left over.

> __Bridge: the Capital Asset Pricing Model (CAPM).__ L5b ended with the market portfolio. The CAPM is an equilibrium statement about expected __excess simple returns__: under its assumptions, an asset's expected excess return equals its beta times the market's expected excess return, with no intercept. Written in our annualized growth-rate observations with a risk-free growth rate $g_{f}$, the CAPM-style relation is $g_{i,t}-g_{f}\approx\beta_{i}\,(g_{M,t}-g_{f})+\varepsilon_{i,t}$, an approximation rather than an identity, and a time-series regression of that form keeps an intercept, whose CAPM value would be zero. We will not test the CAPM here; we borrow its one-factor structure.

Factor models can also carry more than one factor, and the best known does.

> __Example: the Fama-French three-factor model.__ The most widely used multi-factor model adds two factors to the market: a size factor (the return of small-capitalization stocks minus large-capitalization stocks, SMB) and a value factor (the return of high book-to-market stocks minus low book-to-market stocks, HML), written in returns as $r_{i,t}=\alpha_{i}+\beta_{i}r_{M,t}+s_{i}\,\text{SMB}_{t}+h_{i}\,\text{HML}_{t}+\varepsilon_{i,t}$ ([Fama and French, 1992](https://doi.org/10.1111/j.1540-6261.1992.tb04398.x); [Fama and French, 1993](https://doi.org/10.1016/0304-405X(93)90023-5)). Its extra factors explain more of the variation of diversified portfolios than the market alone, at the cost of more parameters to estimate.

Today we keep only the market factor. That is the __single index model__.
___

## Single Index Models (SIMs)
The single index model, introduced by Sharpe as a simplified model for portfolio analysis ([Sharpe, 1963](https://doi.org/10.1287/mnsc.9.2.277)), keeps the market factor and drops the others. For asset $i\in\mathcal{P}$ on trading days $t=1,2,\dots,N$, with an intercept $\alpha_{i}$ (units: inverse years), a market exposure $\beta_{i}$ (dimensionless), and a residual $\varepsilon_{i,t}$ (units: inverse years), the model is:
$$
\boxed{
\begin{align*}
g_{i,t} &= \alpha_{i} + \beta_{i}\,g_{M,t} + \varepsilon_{i,t}\qquad i\in\mathcal{P},\quad t=1,2,\dots,N
\end{align*}}
$$
The intercept $\alpha_{i}$ is the asset's expected growth rate when the market's is zero, the part of its average growth not attributable to market exposure; the residual $\varepsilon_{i,t}$ is the __idiosyncratic__ (asset-specific) part of the day's growth rate, the part the market does not explain. The population assumptions that make the model a covariance model are:
$$
\begin{align*}
\mathbb{E}\left[\varepsilon_{i,t}\right] = 0,\qquad
\text{Cov}\left(\varepsilon_{i,t},g_{M,t}\right) = 0,\qquad
\text{Cov}\left(\varepsilon_{i,t},\varepsilon_{j,t}\right) = 0\quad(i\neq j)
\end{align*}
$$
The first two define $\alpha_{i}$ and $\beta_{i}$ as the coefficients of the population least-squares projection of $g_{i}$ on $g_{M}$; they cost nothing. The third is an __additional restriction__: it says the market is the only source of co-movement between assets, so that once the market's effect is removed, nothing is left in common. Fitting the model separately for every asset does nothing to make it true, and it is the assumption L6b's covariance model rests on, so we will check it in the example. In this section $\mathbb{E}$, $\text{Var}$, and $\text{Cov}$ refer to a single trading day, and we drop the $t$ subscript when it is not needed.

> __Aside: return versus growth.__ Sharpe's original model was written in one-period returns. With the log return $r_{i,t}=\Delta{t}\,g_{i,t}$ (L5a), the return form $r_{i,t}=\alpha_{r,i}+\beta_{i}\,r_{M,t}+\varepsilon_{r,i,t}$ and the growth form above are the same model with $\alpha_{r,i}=\Delta{t}\,\alpha_{i}$, $\varepsilon_{r,i,t}=\Delta{t}\,\varepsilon_{i,t}$, and the __same__ $\beta_{i}$: divide the return form by $\Delta{t}$ and the growth form appears. Beta is convention-free; the intercept and the residual scale with the time step. We work in growth-rate units throughout, and every variance below is a growth-rate variance.

### What do the parameters mean?
Taking the covariance of both sides of the SIM with $g_{M}$ and using $\text{Cov}(\varepsilon_{i},g_{M})=0$ (the intercept is a constant) gives $\text{Cov}(g_{i},g_{M})=\beta_{i}\,\text{Var}(g_{M})$. Write $\sigma_{g,M}=\sqrt{\text{Var}(g_{M})}$ and $\sigma_{g,i}=\sqrt{\text{Var}(g_{i})}$ for the market's and the asset's growth-rate standard deviations (L3a's $\sigma_{g}$; not volatilities) and $\rho_{iM}$ for the correlation of the two growth rates; whenever both standard deviations are positive:
$$
\boxed{
\begin{align*}
\beta_{i} &= \frac{\text{Cov}\left(g_{i},g_{M}\right)}{\text{Var}\left(g_{M}\right)} = \rho_{iM}\,\frac{\sigma_{g,i}}{\sigma_{g,M}}\quad\blacksquare
\end{align*}}
$$
Beta is the slope of the projection: a beta of $1.4$ means that a market move of one unit comes with an expected asset move of $1.4$ units, not that the asset always moves by that much, because the intercept and the residual are there on every day. The intercept is then fixed by the means, $\alpha_{i}=\mathbb{E}[g_{i}]-\beta_{i}\,\mathbb{E}[g_{M}]$.

Taking the variance of both sides and using only that $\varepsilon_{i}$ and $g_{M}$ have zero covariance (independence is not needed), with $\sigma^{2}_{g,\varepsilon,i}=\text{Var}(\varepsilon_{i})$ the residual growth-rate variance, splits the asset's risk in two:
$$
\boxed{
\begin{align*}
\underbrace{\text{Var}\left(g_{i}\right)}_{\text{total}} &= \underbrace{\beta_{i}^{2}\,\sigma^{2}_{g,M}}_{\text{systematic}} + \underbrace{\sigma^{2}_{g,\varepsilon,i}}_{\text{idiosyncratic}}\quad\blacksquare
\end{align*}}
$$
The __systematic__ part is inherited from the market and cannot be diversified away by adding more assets that share the same market; the __idiosyncratic__ part is specific to the asset. The two do not have to be comparable: a stock can have a beta well above one (a lot of systematic exposure) and still be dominated by its idiosyncratic variance, and an index fund can have a beta near one and almost no idiosyncratic variance. So "beta greater than one" means "amplifies market movements", not "more volatile than the market". The fraction of the asset's variance the market explains is $\beta_{i}^{2}\sigma^{2}_{g,M}/\text{Var}(g_{i})=\rho_{iM}^{2}$, which is what the coefficient of determination of the fitted regression estimates (for an asset whose growth rate is not constant).

> __Beta as a dial:__ $\beta_{i}=1$ moves with the market one for one; $\beta_{i}>1$ amplifies market moves; $0<\beta_{i}<1$ dampens them; $\beta_{i}<0$ moves against the market. In our dataset the estimated betas of the S&P 500 firms run from near zero to a little above two around a median close to one.

Wow, that is a lot from one line. But how do we estimate the parameters, and what does the model buy us for a portfolio? First the portfolio, because that is why we are here.
___

## The Covariance Implied by a SIM
Collect the assets of $\mathcal{P}$ into vectors: growth rates $\mathbf{g}$, intercepts $\boldsymbol{\alpha}$, betas $\boldsymbol{\beta}$, and residuals $\boldsymbol{\varepsilon}$, each with one entry per asset. The SIM for every asset at once is the vector model:
$$
\begin{align*}
\mathbf{g} &= \boldsymbol{\alpha} + \boldsymbol{\beta}\,g_{M} + \boldsymbol{\varepsilon}
\end{align*}
$$
Under the three assumptions, with $\bar g_{M}=\mathbb{E}[g_{M}]$, $\sigma^{2}_{g,M}=\text{Var}(g_{M})$, and $\mathbf{D}_{g}=\text{diag}(\sigma^{2}_{g,\varepsilon,1},\sigma^{2}_{g,\varepsilon,2},\dots)$ the diagonal matrix of residual variances, the mean and covariance of the growth-rate vector are:
$$
\boxed{
\begin{align*}
\bar{\mathbf{g}} = \mathbb{E}\left[\mathbf{g}\right] &= \boldsymbol{\alpha}+\boldsymbol{\beta}\,\bar g_{M}\\
\mathbf{\Sigma}_{g} = \text{Cov}\left(\mathbf{g}\right) &= \sigma^{2}_{g,M}\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top} + \mathbf{D}_{g}\quad\blacksquare
\end{align*}}
$$
so that $\text{Cov}(g_{i},g_{j})=\beta_{i}\beta_{j}\sigma^{2}_{g,M}$ for $i\neq j$ and $\text{Var}(g_{i})=\beta_{i}^{2}\sigma^{2}_{g,M}+\sigma^{2}_{g,\varepsilon,i}$ on the diagonal, the decomposition of the previous section. The first term has rank at most one (exactly one unless every beta is zero): every off-diagonal covariance is a product of two betas times one number. This is the compression: instead of one covariance per pair of assets, $|\mathcal{P}|(|\mathcal{P}|+1)/2$ numbers, the SIM needs one beta and one residual variance per asset plus the market variance, $2|\mathcal{P}|+1$ numbers (and, for the mean vector, one intercept per asset plus the market mean, $|\mathcal{P}|+1$ more). For the S&P 500 that is on the order of a thousand parameters against more than a hundred thousand.

> __What is assumed away:__ The off-diagonal zeros of $\mathbf{D}_{g}$ are the third assumption. If the residuals of two assets are correlated (two banks, two chip makers, two share classes of the same firm), their true covariance is $\beta_{i}\beta_{j}\sigma^{2}_{g,M}$ __plus__ the residual covariance the model sets to zero. The estimation example measures the residual correlations across our dataset: most pairs are close to zero, related firms keep a few tenths, and near-duplicates are almost perfectly correlated. Those are the errors L6b's covariance model inherits.

### Portfolio risk under a SIM
For fixed weights $\mathbf{w}$, the linearized portfolio growth rate $g_{p}=\mathbf{w}^{\top}\mathbf{g}$ (a one-period statistic, as in L5b) inherits the model's structure with a portfolio beta $\beta_{p}=\mathbf{w}^{\top}\boldsymbol{\beta}$ and a portfolio residual $\varepsilon_{p}=\mathbf{w}^{\top}\boldsymbol{\varepsilon}$:
$$
\boxed{
\begin{align*}
g_{p} &= \mathbf{w}^{\top}\boldsymbol{\alpha}+\beta_{p}\,g_{M}+\varepsilon_{p},\qquad
\text{Var}\left(g_{p}\right) = \underbrace{\beta_{p}^{2}\,\sigma^{2}_{g,M}}_{\text{market risk}}+\underbrace{\mathbf{w}^{\top}\mathbf{D}_{g}\mathbf{w}}_{\text{residual risk}}
\end{align*}}
$$
The market term is common-factor risk: it depends on the weights only through the portfolio beta $\beta_{p}$, and adding more assets with the same market exposure does not reduce it (a fully invested portfolio escapes it only by driving $\beta_{p}$ to zero). The residual term can be diversified: for equal weights $w_{i}=1/|\mathcal{P}|$ over $|\mathcal{P}|$ assets whose residual variances are bounded by a constant $K$, it is $\frac{1}{|\mathcal{P}|^{2}}\sum_{i\in\mathcal{P}}\sigma^{2}_{g,\varepsilon,i}\leq K/|\mathcal{P}|$, which shrinks as the portfolio broadens. Two conditions carry the conclusion: the weights are fixed and spread out (a concentrated portfolio does not average its residuals away), and the residuals are uncorrelated (with a general residual covariance $\mathbf{\Omega}_{\varepsilon}$ the second term is $\mathbf{w}^{\top}\mathbf{\Omega}_{\varepsilon}\mathbf{w}$, and correlated residuals do not vanish as the portfolio broadens). Under those conditions the SIM says something L5a's full covariance could not say so simply: diversification removes idiosyncratic risk and leaves the market risk set by the portfolio's beta.
___

## Estimation of SIM Parameters
We estimate the SIM from historical data by least squares, one asset at a time. Suppose we have $N$ trading days of aligned growth-rate observations of asset $i$ and the market, exactly the columns of the L5a data matrix. Pack the asset's growth rates into the response vector $\mathbf{y}=(g_{i,1},\dots,g_{i,N})^{\top}$ and build the __design matrix__ $\hat{\mathbf{X}}\in\mathbb{R}^{N\times2}$ (the regression design matrix of L4b) whose first column is ones, for the intercept, and whose second column holds the market growth rates $g_{M,1},\dots,g_{M,N}$. With the parameter vector $\boldsymbol{\theta}_{i}=(\alpha_{i},\beta_{i})^{\top}$ and the residual vector $\boldsymbol{\varepsilon}=(\varepsilon_{i,1},\dots,\varepsilon_{i,N})^{\top}$, all $N$ observations of the SIM at once read:
$$
\begin{align*}
\mathbf{y} &= \hat{\mathbf{X}}\,\boldsymbol{\theta}_{i} + \boldsymbol{\varepsilon}
\end{align*}
$$
The __least-squares__ estimate is the parameter vector that minimizes the sum of squared residuals, and when $\hat{\mathbf{X}}$ has full column rank (the market growth rates are not all equal) it is unique and satisfies the normal equations:
$$
\boxed{
\begin{align*}
\hat{\boldsymbol{\theta}}_{i} = \arg\min_{\boldsymbol{\theta}}\;\lVert\mathbf{y}-\hat{\mathbf{X}}\boldsymbol{\theta}\rVert_{2}^{2}
\quad\Longrightarrow\quad
\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)\hat{\boldsymbol{\theta}}_{i} = \hat{\mathbf{X}}^{\top}\mathbf{y}
\quad\Longrightarrow\quad
\hat{\boldsymbol{\theta}}_{i} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}\quad\blacksquare
\end{align*}}
$$
The fitted values are $\hat{\mathbf{y}}=\hat{\mathbf{X}}\hat{\boldsymbol{\theta}}_{i}$ and the residuals are $\mathbf{r}=\mathbf{y}-\hat{\mathbf{y}}$; the sample versions of the projection formulas of the previous section fall out: $\hat{\beta}_{i}$ is the sample covariance of the asset and market growth rates over the sample variance of the market's, and $\hat{\alpha}_{i}=g^{\prime}_{i}-\hat{\beta}_{i}g^{\prime}_{M}$ with $g^{\prime}$ the sample means. In code we do not form the inverse: a QR-based solve (`X̂ \ y`) or the singular value decomposition computes the same estimate more stably, and the example does it all three ways to show they agree. Regularized (ridge) versions of the estimator, which shrink the parameters toward zero and change their sampling distribution, are treated in the optional advanced material.

> __Example__
>
> [▶ Estimate single index models from historical data](CHEME-5660-L6a-Example-SVD-SIM-Estimation-Fall-2026.ipynb). Build the growth-rate matrix from the 2014 to 2024 data, fit the SIM for one firm by least squares three equivalent ways, compute its classical uncertainty and fit statistics next to those of an index fund, then fit every security in the dataset, check the residual-correlation assumption the SIM covariance relies on, and save the parameter archive that L6b builds portfolios from.

___

## Uncertainty and Diagnostics
An estimate without its uncertainty is a number, not a statement (we drop the asset index on $\sigma^{2}_{g,\varepsilon}$ in this section, since one regression is in view). Under the __classical regression assumptions__, the design $\hat{\mathbf{X}}$ held fixed and the residuals with zero conditional mean, $\mathbb{E}[\boldsymbol{\varepsilon}\,|\,\hat{\mathbf{X}}]=\mathbf{0}$, independent across days, with a common variance, and Gaussian, $\boldsymbol{\varepsilon}\sim\mathcal{N}(\mathbf{0},\sigma^{2}_{g,\varepsilon}\mathbf{I})$, the least-squares estimator is unbiased, $\mathbb{E}[\hat{\boldsymbol{\theta}}_{i}]=\boldsymbol{\theta}_{i}$, and Gaussian with covariance $\sigma^{2}_{g,\varepsilon}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}$ (substitute $\mathbf{y}=\hat{\mathbf{X}}\boldsymbol{\theta}_{i}+\boldsymbol{\varepsilon}$ into the estimator: $\hat{\boldsymbol{\theta}}_{i}=\boldsymbol{\theta}_{i}+(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}\hat{\mathbf{X}}^{\top}\boldsymbol{\varepsilon}$, a linear transformation of a Gaussian vector). The residual growth-rate variance is estimated from the residuals, and the observations are already growth rates, so no time-step factor enters:
$$
\boxed{
\begin{align*}
s^{2}_{g,\varepsilon} = \frac{\lVert\mathbf{r}\rVert_{2}^{2}}{N-2},\qquad
\widehat{\text{Cov}}\left(\hat{\boldsymbol{\theta}}_{i}\right) = s^{2}_{g,\varepsilon}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1},\qquad
\text{SE}\left(\hat{\theta}_{j}\right) = \sqrt{\left[\widehat{\text{Cov}}\left(\hat{\boldsymbol{\theta}}_{i}\right)\right]_{jj}}
\end{align*}}
$$
where the two in the denominator counts the estimated parameters, and a $95\%$ confidence interval for either parameter is $\hat{\theta}_{j}\pm t_{0.975,N-2}\,\text{SE}(\hat{\theta}_{j})$, with $t_{0.975,N-2}$ the Student $t$ quantile with $N-2$ degrees of freedom (with eleven years of daily data, $1.96$ to three digits). The interval for $\hat{\beta}_{i}$ is usually narrow, because the market column has a lot of variation; the interval for $\hat{\alpha}_{i}$ is usually wide relative to its value, and whether it covers zero says whether the data can tell the asset's average growth from what its market exposure alone would deliver. In L4b's units, the residual's diffusion volatility is $\sqrt{\Delta{t}}\,s_{g,\varepsilon}$.

### How good is the fit?
The __coefficient of determination__ is the fraction of the asset's growth-rate variation the fitted market term explains:
$$
\begin{align*}
R^{2} &= 1 - \frac{\lVert\mathbf{r}\rVert_{2}^{2}}{\sum_{t=1}^{N}\left(g_{i,t}-g^{\prime}_{i}\right)^{2}} = \hat{\rho}_{iM}^{2}
\end{align*}
$$
where the second equality holds for a least-squares regression with an intercept and a non-constant response, and it is the sample version of $\rho_{iM}^{2}$ from the risk decomposition. An index fund can have $R^{2}$ near $0.85$ against SPY while a single stock with a larger beta sits near $0.25$; both are consistent with the model.

### What daily data do to the assumptions
Daily growth rates violate the classical list (L3a): the residuals are heavy-tailed rather than Gaussian, their absolute values cluster in time, and in our dataset the growth rates themselves carry a positive autocorrelation at lag one because they are built from volume-weighted average prices, which average over the trading day. The classical intervals are therefore optimistic, and for beta materially so: an autocorrelation-consistent standard error for the firm in the examples is roughly forty to one hundred percent larger than the classical one, depending on how many lags it uses. Two remedies are standard. A __bootstrap__ builds many synthetic datasets from the fitted model, $\mathbf{y}^{(b)}=\hat{\mathbf{X}}\hat{\boldsymbol{\theta}}_{i}+\boldsymbol{\varepsilon}^{(b)}$, re-estimates the parameters on each, and reads their spread: the __parametric__ bootstrap draws $\boldsymbol{\varepsilon}^{(b)}$ from $\mathcal{N}(\mathbf{0},s^{2}_{g,\varepsilon}\mathbf{I})$ (the classical model, checked by simulation), the __empirical-residual__ bootstrap resamples the fitted residuals themselves and so keeps their tails; both draw days independently. When the dependence matters, heteroskedasticity- and autocorrelation-consistent (Newey-West) standard errors or a block bootstrap, which resamples stretches of consecutive days, are the credible alternatives. The second example runs both bootstraps, audits the residuals, and computes the Newey-West standard errors.

> __Example__
>
> [▶ Bootstrap uncertainty in a single index model](CHEME-5660-L6a-Example-SIM-Parameter-Uncertainty-Fall-2026.ipynb). Fit the SIM with the course package, resample it two ways (empirical residuals and Gaussian innovations), compare the bootstrap intervals with the classical standard errors, audit the residuals for the heavy tails and dependence that every independent-draws method ignores, and compute the autocorrelation-consistent standard errors that dependence calls for.

___

## What R-squared, Alpha, and Beta Do Not Prove
Three cautions before the estimates go into an optimizer. The coefficient of determination is the sample fraction of variation explained by the fitted market term; it is not the probability that the model is true, not a forecast-accuracy guarantee, and not a measure of investment quality: a low $R^{2}$ can come with a precisely estimated beta, and a high $R^{2}$ with unstable parameters. A positive estimated alpha is an intercept conditional on the market proxy, the sample window, the price series used (our growth rates come from volume-weighted average prices, not dividend-adjusted total returns), and the specification; it is not abnormal performance, and its confidence interval usually covers zero. Beta is exposure to the chosen index and sampling interval, not total risk. Before a SIM covariance goes into L6b's optimizer, inspect the residual cross-correlations, the stability of the parameters across windows, and the sensitivity to the market proxy; and remember that the growth-rate, log-return, and covariance-rate conventions of L5a differ by powers of $\Delta{t}$, so every input must use the same one.
___

## Optional Advanced Material
The notebook below extends today's material. It is optional and is not a prerequisite for L6b; the [advanced index](advanced/README.md) describes it.

* [▶ Single index model estimation theory](advanced/sim/CHEME-5660-L6a-Advanced-SIM-Theory-Fall-2026.ipynb). Derive the ridge (regularized) estimator and its covariance, distinguish the residual and parametric bootstraps, keep the growth-rate, log-return, and volatility conventions straight, propagate parameter uncertainty into portfolio risk and weights, and write the maximum-Sharpe problem as a second-order cone program.
___

## Summary
In this lecture, we introduced the single index model, which explains each asset's growth rate by its exposure to a market index plus an idiosyncratic residual, derived the covariance matrix and the portfolio risk it implies, and estimated its parameters by least squares with their classical uncertainty and diagnostics.

> __Key Takeaways:__
>
> * **One factor splits risk in two:** Under the single index model an asset's growth-rate variance is its beta squared times the market variance plus a residual variance, so beta measures systematic exposure while the coefficient of determination measures how much of the variation the market explains, and the two can disagree.
> * **The SIM covariance is rank one plus diagonal, and that rests on an assumption:** The model needs one beta and one residual variance per asset plus the market variance instead of one covariance per pair, and a broad equally weighted portfolio diversifies its residual risk away and keeps only its beta times the market risk, provided the residuals of different assets are uncorrelated, which fitting the regressions separately does not guarantee.
> * **Least squares gives the estimates, and daily data qualify the uncertainty:** The parameters come from a two-column regression solved without forming an inverse, their classical standard errors and intervals are in growth-rate units with no time-step factor, and heavy tails, dependence at lag one, and volatility clustering make bootstrap or robust intervals the credible complement.

Next time, we put the SIM mean vector and covariance matrix into the minimum-variance allocation problems of L5b, first with risky assets only and then with the risk-free asset and the capital allocation line.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___